In [107]:
# Install ART
!pip install adversarial-robustness-toolbox

In [108]:
# Connecting to the Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [109]:
# Checking necessary packages are installed

import torch, art, pandas, sklearn
print(torch.__version__, art.__version__, pandas.__version__, sklearn.__version__)

2.14.1+cpu 1.20.1 2.2.3 1.6.1


In [110]:
#the path to the training file is /content/drive/MyDrive/nslkdd/KDDTrain+.txt



In [111]:
#column names

cols = ["duration","protocol_type","service","flag","src_bytes","dst_bytes",
"land","wrong_fragment","urgent","hot","num_failed_logins","logged_in",
"num_compromised","root_shell","su_attempted","num_root","num_file_creations",
"num_shells","num_access_files","num_outbound_cmds","is_host_login",
"is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
"rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
"srv_diff_host_rate","dst_host_count","dst_host_srv_count",
"dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
"dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
"dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"]

In [112]:
#import

import pandas as pd
import os


In [113]:
folder = "/content/drive/MyDrive/nslkdd/"
os.listdir(folder)

['KDDTest+.txt', 'KDDTrain+.txt', 'baseline_model.pt', 'robust_model.pt']

**Source**

The NSL-KDD dataset was downloaded from Zenodo on the 18 November 2026. Citation
Saad. (2025). NSL-KDD Dataset [Data set]. Zenodo. https://doi.org/10.5281/zenodo.17424143   second citation: Tavallaee, M., Bagheri, E., Lu, W., & Ghorbani, A. A. (2009, July). A detailed analysis of the KDD CUP 99 data set. In 2009 IEEE symposium on computational intelligence for security and defense applications (pp. 1-6).IEEE.
**Size**
 The training dataset has 125973 rows and 43 columns, The test dataset has 22544 rows and 43 columns. One major issue with the dataset was that values were separated with tab instead of comma hence read_csv was looking for commas and when it could not see it values were lumped together into the first column. to separate the values we had to use tab (\t) as a value for sep.

**Class balance**

 There are 22 attack and one normal for the training labels while the test labels were already reduced to Attack and Normal. Normal has 53.5% while attack has 46.5% for training proportion. Normal has 43.1%  while attack has 56.9% for testing proportion. labels were converted to lowercase before comparing them with “normal”.

 **Observations**

  Normal is more common in the training dataset than attack while attack is more common in the test data set than normal. The model learns from the training dataset, the model may lean towards guessing “normal”, so it may miss attacks in the test set.
  The label and difficulty are not included. Real traffic does not have parameters to communicate with the model how hard it is to classify. The score was included because other researchers have tested this dataset on other models. A model that depends on this will perform well on this dataset but badly on real traffic. is_attack was derived from label hence the label contains the answer. The model must not be given label as an input. With it, the model would score almost perfectly on this dataset just by reading the answer but would be useless on real traffic.  This problem is known as leakage

In [114]:
# load the training file

train = pd.read_csv(folder + "KDDTrain+.txt", header=None, names= cols, sep="\t")


In [115]:
## Show the size of the training data as (rows, columns)
train.shape

(125973, 43)

In [116]:
# Show the first five rows to check the data loaded correctly
train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [117]:
#select label column and list each different value once, without repeating any value twice
train['label'].unique()

array(['normal', 'neptune', 'warezclient', 'ipsweep', 'portsweep',
       'teardrop', 'nmap', 'satan', 'smurf', 'pod', 'back',
       'guess_passwd', 'ftp_write', 'multihop', 'rootkit',
       'buffer_overflow', 'imap', 'warezmaster', 'phf', 'land',
       'loadmodule', 'spy', 'perl'], dtype=object)

In [118]:
#load the test file

test = pd.read_csv(folder + "KDDTest+.txt", header=None, names= cols, sep="\t")
print(test.shape)


(22544, 43)


In [119]:
train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [120]:
# Create a binary column: 1 = attack, 0 = normal
train['is_attack'] = (train['label'] != 'normal').astype(int)
test['is_attack'] = (test['label']  != 'normal').astype(int)

In [121]:
# Check the new column against the original labels

train[['label', 'is_attack']].head(10)

,label,is_attack
0,normal,0
1,normal,0
2,neptune,1
3,normal,0
4,normal,0
5,neptune,1
6,neptune,1
7,neptune,1
8,neptune,1
9,neptune,1


In [122]:
# Look at the exact label values in the test file
test['label'].unique()

array(['Attack', 'Normal'], dtype=object)

In [123]:
#Ensuring lowercase due to python case sensitivty

train['is_attack'] = (train['label'].str.lower() != 'normal').astype(int)
test['is_attack'] = (test['label'].str.lower() != 'normal').astype(int)

In [124]:
test[['label', 'is_attack']].head(10)

,label,is_attack
0,Attack,1
1,Attack,1
2,Normal,0
3,Attack,1
4,Attack,1
5,Normal,0
6,Normal,0
7,Attack,1
8,Normal,0
9,Attack,1


In [125]:
# Show the share of normal (0) and attack (1) rows in the training data
train['is_attack'].value_counts(normalize=True)


,proportion
is_attack,
0,0.534583
1,0.465417


In [126]:
# Show the share of normal (0) and attack (1) rows in the test data
test['is_attack'].value_counts(normalize=True)

,proportion
is_attack,
1,0.569242
0,0.430758


## Preparing the inputs

**Inputs and target:** difficulty, and labels were dropped to prevent leakage,  The target is_attack, was dropped from input.

**Encoding:** One-hot encoding was used to spit the protocol_type column. Also, service and flag were encoded. This increased the number of columns in the training dataset to 122 making it more than what's in the testing dataset with 116 columns. Some values, mostly services, appear in train but never in test, so `get_dummies` made columns for them in train only. This resulted in some columns appearing in training dataset but absent in testing data set. A model expects the same number of columns and same order in both data sets. To fix this , reindex(columns=X_train.columns, fill_value=0) was used to create the same number of columns in X_test. Both datasets ended up with 122 columns each.

**Scaling:** To ensure columns with big values don't drown columns with smaller values. In the attack stage, a small change should mean the same size of change on every feature.. MinMaxScaler was used to force values into the range of 0 to 1. fit the scaler on training data only, then apply the same scaling to test data, as you would to any new traffic. X_test used transform instead of fit_transform because the minimum and maximum traffic cannot be determined ahead of time. So that results reflect how the model would perform in real use.

**Observations:** The test set, scaled with the training min and max. Maximum value returned as 2.5 which is more than 1 after scaling showing some traffics may contain extreme values above what is being used to train the model. Test contains traffic, including attack types, that the model never sees in training.

In [127]:
# List the columns that contain text instead of numbers
train.select_dtypes(include='object').columns

Index(['protocol_type', 'service', 'flag', 'label'], dtype='object')

In [128]:
# Separate inputs (X) from the answer (y), leaving out label and difficulty to avoid leakage
X_train = train.drop(columns=['label', 'difficulty', 'is_attack'])
y_train = train['is_attack']

X_test = test.drop(columns=['label', 'difficulty', 'is_attack'])
y_test = test['is_attack']

# Check the number of rows and input columns in each set
print(X_train.shape)
print(X_test.shape)

(125973, 41)
(22544, 41)


In [129]:
# One-hot encode the text columns so the model gets numbers, not words
X_train = pd.get_dummies(X_train, columns=['protocol_type', 'service', 'flag'])
X_test = pd.get_dummies(X_test, columns=['protocol_type', 'service', 'flag'])

# Check the new number of columns
print(X_train.shape)
print(X_test.shape)

(125973, 122)
(22544, 116)


In [130]:
# Give the test set exactly the same columns as the training set, in the same order (missing ones filled with 0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Check the new number of columns
print(X_train.shape)
print(X_test.shape)

(125973, 122)
(22544, 122)


In [131]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn min/max from X_train then rescale X_train
X_test_scaled = scaler.transform(X_test)       # rescale  X_test using X_train's min/max

# Confirm the training data now sits between 0 and 1
print(X_train_scaled.min(), X_train_scaled.max())

0.0 1.0


In [132]:
# Check the range of the test data after scaling with the training min/max
print(X_test_scaled.min(), X_test_scaled.max())

0.0 2.5


## Building and training the model

**Data format:** Data was converted to tensors. Inputs must be decimals to conform with PyTorch's standard, input tensors are in torch.float32. The targets are class labels in 0 or 1 and the tensors are whole numbers in torch.long format.

**Model:** A small neural network with nn.Sequential was built. The model uses layer sizes 122, 64, 32, and 2 with ReLU in between the layers to learn the pattern. 122 is the input while 2 is the number of output scores. 64 and 32 were my choices, while 122 and 2 were fixed by the data and the question.

**Training:** Using the loss function it compares the model's 2 scores with the correct answer and gives back one number. The optimiser adjusts the model's setting with model.parameters(). The Learning rate is 0.001. The loss says how wrong; the optimiser fixes the parameters. `CrossEntropyLoss` and Adam optimiser are used. Batch size is 256, epochs are 5 and seed is 42. The training loss fell from 0.1192 to 0.0239 over 5 epochs.

**Results:** Test accuracy is 79.0% and detection rate is 68.7%.

**Observations:** The test accuracy and detection rates are low. From the results, detection is lower than accuracy. This makes the model prone to missing attacks more than it misjudges the normal traffic. The low values can be attributed to the test dataset containing attack types that don’t appear in training at all.. Also, the test value was higher than 1 (up to 2.5) which was larger than the training set. The test accuracy shifted slightly after retraining, result ranges from 78.96% to 79.00%.

In [133]:
import torch
# Convert inputs and targets into PyTorch tensors (decimals for X, whole numbers for y)
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

print(X_train_t.shape)
print(y_train_t.dtype)

torch.Size([125973, 122])
torch.int64


In [134]:
# Import PyTorch's neural network tools under the short name nn
import torch.nn as nn

# Fix the random starting values so results are the same every run
torch.manual_seed(42)

# Define the model: a small network that funnels 122 features down to 2 scores
model = nn.Sequential(
    nn.Linear(122, 64),   # Layer 1: takes the 122 input features, outputs 64 numbers
    nn.ReLU(),            # Lets the network learn non-straight-line patterns
    nn.Linear(64, 32),    # Layer 2: takes 64 numbers, outputs 32
    nn.ReLU(),            # Same as above, between layers 2 and 3
    nn.Linear(32, 2)      # Output layer: 2 scores, one for normal and one for attack
)

# Test the model on the first 5 rows to check the output shape
out = model(X_train_t[:5])
print(out.shape)

torch.Size([5, 2])


In [135]:
# Loss function: measures how wrong the model's guesses are (lower is better)
loss_fn = nn.CrossEntropyLoss()

# Optimiser: adjusts the model's settings to reduce the loss, in steps of size lr
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(loss_fn)
print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [136]:
# Load the saved baseline model from Drive (use this in place of retraining after a reset)
model.load_state_dict(torch.load(folder + "baseline_model.pt"))

<All keys matched successfully>

In [137]:
from torch.utils.data import TensorDataset, DataLoader

# Pair each row of inputs with its correct answer
train_ds = TensorDataset(X_train_t, y_train_t)

# Serve the training data in shuffled batches of 256 rows
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

# Number of batches per pass through the training data
print(len(train_loader))

493


In [138]:

#training loop
# DISABLED: model is now loaded from Drive. Remove the # signs to retrain.

# torch.manual_seed(42)
# epochs = 5

# for epoch in range(epochs):
#     model.train()                      # put the model in training mode
#     total_loss = 0

#     for xb, yb in train_loader:        # xb = batch of inputs, yb = batch of answers
#         optimizer.zero_grad()             # clear old nudges
#         out = model(xb)               # guess
#         loss = loss_fn(out, yb)         # measure how wrong
#         loss.backward()                # work out the nudges
#         optimizer.step()                  # apply the nudges
#         total_loss += loss.item()

#     avg_loss = total_loss / len(train_loader)
#     print(f"Epoch {epoch + 1}: average loss = {avg_loss:.4f}")

In [139]:
# Switch to testing mode and make predictions on the test set
model.eval()
with torch.no_grad():
    test_out = model(X_test_t)         # the network, given the test inputs
    preds = test_out.argmax(dim=1)     # pick the higher score: 0 = normal, 1 = attack

# Accuracy: share of predictions that match the true answers
accuracy = (preds == y_test_t).float().mean().item()
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.7900


In [140]:
# Detection rate: of the real attacks, what share did the model flag as attack?

# Mark every test row that is truly an attack (True) or normal (False)
attack_rows = (y_test_t == 1)

# Keep only the real attacks, check which ones the model called attack, and take the average
detection_rate = (preds[attack_rows] == 1).float().mean().item()

# Show the result to four decimal places
print(f"Detection rate: {detection_rate:.4f}")

Detection rate: 0.6873


## Attacking the Model

TCP is coded as 1 for  TCP and 0 for not TCP, there is no in between. So, 1.1 does not fit into the scale for determining whether it is TCP or not. It is easier to changer numbers than to change network traffic thereby creating a research gap. What works perfectly in the feature space may not be obtainable in the problem space.

To make the simulated attack realistic, only features which an attacker can change in problem space are accessible. Features with the following prefix protocol_type_, service_, flag were locked. Out of the 41 original features. The 3 text columns  became 84 one-hot columns which were locked. The remaining 38 number columns stayed open to the attack. 84 one-hot columns were locked and the detection rate increased from 30.26% to 46.51%.  The increase in the detection can be attributed to 84 columns locked thereby reducing the number of features to change which fooled the model less often. The 46.51% is more trustworthy than 30.26% detection because features that the attacker cannot change in problem space have been locked. The gap between 46.51% and 68.73% is still wide, the real-world effect probably lies somewhere between those two figures. Amongst the 38 features opened some rules are yet to be enforced such as 0 or 1 only, derived numbers and integers, The threat may still overstated at this point.

In [141]:
# Import ART's wrapper for PyTorch models
from art.estimators.classification import PyTorchClassifier

# Wrap the trained model so ART's attacks can use it
classifier = PyTorchClassifier(
    model=model,                 # the trained network from step 3
    loss=loss_fn,                # the same loss function used in training
    optimizer=optimizer,         # the same Adam optimiser used in training
    input_shape=(122,),          # each row has 122 input features
    nb_classes=2,                # two classes: normal (0) and attack (1)
    clip_values=(0.0, float(X_test_scaled.max()))  # attacks can't push values below 0 or above the test maximum (2.5)
)

# Confirm the wrapped model gives the same test accuracy as before
art_preds = classifier.predict(X_test_t.numpy()).argmax(axis=1)  # ART's predictions, as 0 or 1
print((art_preds == y_test_t.numpy()).mean())                     # share that match the true answers

0.7899662881476224


In [142]:
# DISABLED: model is now loaded from Drive. Remove the # signs to retrain.
# # Save the trained model's learned numbers to Drive so it can be reloaded without retraining
# torch.save(model.state_dict(), folder + "baseline_model.pt")
# os.listdir(folder)

In [143]:
# Load the saved baseline model from Drive (use this in place of retraining after a reset)
model.load_state_dict(torch.load(folder + "baseline_model.pt"))

<All keys matched successfully>

In [144]:
# Select only the real attack rows from the test set, as plain arrays for ART
X_attacks = X_test_t.numpy()[attack_rows.numpy()]
y_attacks = y_test_t.numpy()[attack_rows.numpy()]
print(X_attacks.shape)

# Detection rate on these rows before any attack (should match the baseline)
clean_preds = classifier.predict(X_attacks).argmax(axis=1)
print((clean_preds == 1).mean())

(12833, 122)
0.6872905789760773


In [145]:
from art.attacks.evasion import FastGradientMethod

# Set up the FGSM attack on the wrapped model, with a nudge size of 0.1
fgsm = FastGradientMethod(estimator= classifier, eps=0.1)

# Create adversarial versions of the real attack rows
X_adv = fgsm.generate(x=X_attacks, y= y_attacks)

# Detection rate on the adversarial rows
adv_preds = classifier.predict(X_adv).argmax(axis=1)
print((adv_preds == 1).mean())

0.3025792877737084


In [146]:
# Find the position of the protocol_type_tcp column
feature_names = list(X_train.columns)
tcp_idx = feature_names.index('protocol_type_tcp')

# Compare the first 5 attack rows before and after FGSM
print("Before:", X_attacks[:5, tcp_idx])
print("After: ", X_adv[:5, tcp_idx])

Before: [1. 1. 0. 1. 1.]
After:  [1.1 1.1 0.1 1.1 1.1]


In [147]:
import numpy as np

# Build a mask: 0 = locked (one-hot columns), 1 = attack may change it
onehot_prefixes = ("protocol_type_", "service_", "flag_")
mask = np.array([0 if name.startswith(onehot_prefixes) else 1 for name in feature_names]).astype(np.float32)

# How many columns can the attack still change?
print(mask.sum())

38.0


In [148]:
# FGSM again, but only the 38 non-one-hot columns may change
X_adv_masked = fgsm.generate(x=X_attacks, y=y_attacks, mask=mask)

# Detection rate on the masked adversarial rows
masked_preds = classifier.predict(X_adv_masked).argmax(axis=1)
print((masked_preds == 1).mean())

# Confirm the TCP column was left untouched
print("Before:", X_attacks[:5, tcp_idx])
print("After: ", X_adv_masked[:5, tcp_idx])

0.4650510402867607
Before: [1. 1. 0. 1. 1.]
After:  [1. 1. 0. 1. 1.]



## Defence with adversarial training
**Method:** To make the model more resilient and identify attacks disguised as normal, an adversarial version was created with masked FGSM. Disguised attacks example rows are created. The 58,630 masked FGSM training attacks added to the 125,973 original rows, making 184,603. The adversarial training is training the model on disguised attacks which are labelled as attacks.

**Training:** The disguised attack rows are added to the training data set for the defended model to learn from the original training and disguised attack rows. Using np.concatenate([...]) the original training inputs and the adversarial rows are joined into an array. The arrays are converted into tensors. using float32 for inputs and long for answers. Then, the arrays are paired with TensorDataset and serving batches with DataLoader. The combined loader length is 722. The defended model was trained on both clean and disguised data. The defended model used the same two layers (122 → 64 → 32 → 2) and seed set at 42 as the baseline. In 5 epochs, 0.0159 was obtained which is lower than the baseline at 0.0239. The loss at 0.0159 cannot be compared with the baseline of 0.0239 because training with 722 batches per epoch against 493, the defended model made approximately 50% more updates in the 5 epochs.

**Results:** The defended model later tested on clean data, compared with the baseline of 79.00% accuracy and 68.73% the defended model output 79.74 which is higher than the baseline and 70.17% detection rate which is above the baseline. Fresh disguised attack rows with eps =0.1 were created using masked FGSM. The detection became 57.85% higher than the baseline 46.51%. The summary of the result is presented in this table:
| | Baseline | Defended |
|---|---|---|
| Accuracy (clean) | 79.00% | 79.74% |
| Detection, clean attacks | 68.73% | 70.17% |
| Detection, masked FGSM (eps 0.1) | 46.51% | 57.85% |

**Observations:** The baseline of the model dropped from 68.73% to 46.51% and the defended model also dropped from 70.17% to 57.85%, this is 22.22 and 12.32 percentage points respectively. Comparing the two drops, the defended model lost about 55% as much detection as the baseline, so adversarial training reduced the attack's effect by roughly 45%. However, real attacks might be perceived as normal and successfully get through the defence. The defence didn't cost anything on clean data.
**Limitations:** FGSM was the only attack tested with eps set to 0.1, Attacks such as PGD may have a higher effect. Also the disguised training rows were made once against the baseline. This is a single run with one seed.





In [149]:
# Pick out the real attack rows from the training set
train_attack_rows = (y_train_t ==1).numpy()
X_train_attacks = X_train_t.numpy()[train_attack_rows]
y_train_attacks = y_train_t.numpy()[train_attack_rows]

# Create adversarial versions with the same masked FGSM (one-hot columns locked)
X_train_adv = fgsm.generate(x=X_train_attacks, y=y_train_attacks, mask=mask)
print(X_train_adv.shape)

(58630, 122)


In [150]:
# Combine clean training data with the adversarial attack rows
X_combined = np.concatenate([X_train_t.numpy(), X_train_adv])
y_combined = np.concatenate([y_train_t.numpy(), y_train_attacks])

print(X_combined.shape)
print(y_combined.shape)

(184603, 122)
(184603,)


In [151]:
# Convert the combined data into PyTorch tensors (decimals for inputs, whole numbers for answers)
X_combined_t = torch.tensor(X_combined, dtype=torch.float32)
y_combined_t = torch.tensor(y_combined, dtype=torch.long)

# Pair each combined input row with its correct answer
train_adv = TensorDataset(X_combined_t, y_combined_t)

# Serve the combined data in shuffled batches of 256 (new name, so the original train_loader is untouched)
comb_loader = DataLoader(train_adv, batch_size=256, shuffle=True)

# Number of batches per pass through the combined data
print(len(comb_loader))

722


In [152]:
import torch.nn as nn

# Fix the random starting values so results are the same every run
torch.manual_seed(42)

# Define the defended model: same design as the baseline, to be trained on clean plus adversarial data
robust_model = nn.Sequential(
    nn.Linear(122, 64),   # Layer 1: takes the 122 input features, outputs 64 numbers
    nn.ReLU(),            # Lets the network learn non-straight-line patterns
    nn.Linear(64, 32),    # Layer 2: takes 64 numbers, outputs 32
    nn.ReLU(),            # Same as above, between layers 2 and 3
    nn.Linear(32, 2)      # Output layer: 2 scores, one for normal and one for attack
)

In [153]:
# Optimiser for the defended model: adjusts its settings to reduce the loss
robust_optimizer = torch.optim.Adam(robust_model.parameters(), lr=0.001)

In [154]:
torch.manual_seed(42)
epochs = 5

for epoch in range(epochs):
    robust_model.train()                 # put the model in training mode
    total_loss = 0

    for xb, yb in comb_loader:           # xb = batch of inputs, yb = batch of answers
        robust_optimizer.zero_grad()     # clear old nudges
        out = robust_model(xb)           # guess
        loss = loss_fn(out, yb)          # measure how wrong
        loss.backward()                  # work out the nudges
        robust_optimizer.step()          # apply the nudges
        total_loss += loss.item()

    avg_loss = total_loss / len(comb_loader)
    print(f"Epoch {epoch + 1}: average loss = {avg_loss:.4f}")

Epoch 1: average loss = 0.0842
Epoch 2: average loss = 0.0312
Epoch 3: average loss = 0.0236
Epoch 4: average loss = 0.0190
Epoch 5: average loss = 0.0159


In [155]:
# Save the defended model's learned numbers to Drive so it can be reloaded without retraining
# DISABLED: (switch this cell off after running it once, so Run all can't overwrite the file)
#torch.save(robust_model.state_dict(), folder + "robust_model.pt")

# Confirm the file is in the Drive folder
#os.listdir(folder)

In [156]:
# Load the saved defended model from Drive (use this in place of retraining after a reset)
robust_model.load_state_dict(torch.load(folder + "robust_model.pt"))

<All keys matched successfully>

In [157]:
# Switch to testing mode and make predictions on the test set
robust_model.eval()
with torch.no_grad():
    robust_out =robust_model(X_test_t)         # the network, given the test inputs
    robust_preds = robust_out.argmax(dim=1)     # pick the higher score: 0 = normal, 1 = attack

# Accuracy: share of predictions that match the true answers
robust_accuracy= (robust_preds == y_test_t).float().mean().item()
print(f"Robust test accuracy: {robust_accuracy:.4f}")

Robust test accuracy: 0.7974


In [158]:
# Robust detection rate: of the real attacks, what share did the defended model flag as attack?

# Mark every test row that is truly an attack (True) or normal (False)
attack_rows = (y_test_t == 1)

# Keep only the real attacks, check which ones the defended model called attack, and take the average
robust_detection = (robust_preds[attack_rows] == 1).float().mean().item()

# Show the result to four decimal places
print(f"Robust detection rate: {robust_detection:.4f}")

Robust detection rate: 0.7017


In [159]:
# Wrap the defended model so ART's attacks can use it
robust_classifier= PyTorchClassifier(
    model=robust_model,                 # the defended network from step 5
    loss=loss_fn,                # the same loss function used in training
    optimizer=robust_optimizer,         # the defended model's Adam optimiser
    input_shape=(122,),          # each row has 122 input features
    nb_classes=2,                # two classes: normal (0) and attack (1)
    clip_values=(0.0, float(X_test_scaled.max()))  # attacks can't push values below 0 or above the test maximum (2.5)
)

In [160]:
# Set up the same masked FGSM attack, this time against the defended model
robust_fgsm = FastGradientMethod(estimator=robust_classifier, eps=0.1)

# Create adversarial versions of the real test attacks (one-hot columns locked)
X_adv_robust = robust_fgsm.generate(x=X_attacks, y=y_attacks, mask=mask)

# Detection rate of the defended model on these adversarial rows
robust_adv_preds = robust_classifier.predict(X_adv_robust).argmax(axis=1)
print((robust_adv_preds == 1).mean())

# Confirm the TCP column was left untouched
print("Before:", X_attacks[:5, tcp_idx])
print("After: ", X_adv_robust[:5, tcp_idx])

0.5785085326891608
Before: [1. 1. 0. 1. 1.]
After:  [1. 1. 0. 1. 1.]
